In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Gates fracionários

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido com os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Esta página apresenta dois novos tipos de gates suportados pela frota de QPUs IBM Quantum&reg;. Esses gates *fracionários* são suportados em [QPUs Heron](/guides/processor-types#heron) nas seguintes formas:
- $R_{ZZ}(\theta)$ para $0 \lt \theta \leq \pi/2$
- $R_X(\theta)$ para qualquer $\theta$

Esta página discute os casos de uso em que a implementação de gates fracionários pode melhorar a eficiência dos seus fluxos de trabalho, além de como usar esses gates em QPUs IBM Quantum.

## Como usar gates fracionários
Internamente, esses gates fracionários funcionam executando diretamente uma rotação $R_{ZZ}(\theta)$ e $R_X(\theta)$ para um ângulo arbitrário. O uso do gate $R_X(\theta)$ pode reduzir a duração e o erro de rotações de um único qubit com ângulo arbitrário em até um fator de dois. A execução direta da rotação do gate $R_{ZZ}(\theta)$ evita a decomposição em múltiplos objetos `CZGate`, reduzindo igualmente a duração e o erro do circuito. Isso é especialmente útil para circuitos que contêm muitas rotações de um e dois qubits, como ao simular a dinâmica de um sistema quântico ou ao usar um ansatz variacional com muitos parâmetros.

Embora esses tipos de gates estejam na [biblioteca de gates padrão](./circuit-library) que um `QuantumCircuit` pode conter, eles só podem ser usados em QPUs IBM Quantum específicas, e precisam ser carregados com o flag `use_fractional_gates` definido como `True` (mostrado abaixo). Esse flag garantirá que os gates fracionários sejam incluídos no `Target` do backend para o transpilador.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization.timeline import draw as draw_timeline, IQXSimple

from qiskit_ibm_runtime import QiskitRuntimeService


num_qubits = 5
num_time_steps = 3
rx_angle = 0.1
rzz_angle = 0.1

ising_circuit = QuantumCircuit(num_qubits)
for i in range(num_time_steps):
    # rx layer
    for q in range(num_qubits):
        ising_circuit.rx(rx_angle, q)
    for q in range(1, num_qubits - 1, 2):
        ising_circuit.rzz(rzz_angle, q, q + 1)
    # 2nd rzz layer
    for q in range(0, num_qubits - 1, 2):
        ising_circuit.rzz(rzz_angle, q, q + 1)
    ising_circuit.barrier()
ising_circuit.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/9cb7f696-dec0-4393-9320-fe945326c893-0.svg" alt="Output of the previous code cell" />

Este exemplo de código demonstra como usar gates fracionários no contexto de um fluxo de trabalho que simula a dinâmica de uma cadeia de Ising usando gates fracionários. A duração do circuito é então comparada com a de um backend que não usa gates fracionários.

> **Note:** O valor de erro reportado no `Target` de um backend com gates fracionários habilitados é apenas uma cópia do equivalente do gate não fracionário (que pode não ser o mesmo). Isso ocorre porque o reporte de taxas de erro nos gates fracionários ainda não é suportado.

  No entanto, como o tempo de gate dos gates fracionários versus os não fracionários é o mesmo, é uma suposição razoável que suas taxas de erro sejam comparáveis — especialmente quando a principal fonte de erro num circuito é devida ao relaxamento.

In [2]:
service = QiskitRuntimeService()
backend_fractional = service.backend("ibm_torino", use_fractional_gates=True)
backend_conventional = service.backend(
    "ibm_torino", use_fractional_gates=False
)

pm_fractional = generate_preset_pass_manager(
    optimization_level=3, backend=backend_fractional, scheduling_method="alap"
)
pm_conventional = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend_conventional,
    scheduling_method="alap",
)

ising_circuit_fractional = pm_fractional.run(ising_circuit)
ising_circuit_conventional = pm_conventional.run(ising_circuit)

![Saída da célula de código anterior](../docs/images/guides/fractional-gates/extracted-outputs/9cb7f696-dec0-4393-9320-fe945326c893-0.svg)

Especifique dois objetos de backend: um com gates fracionários habilitados e o outro com eles desabilitados, e então transpile os dois.

In [3]:
# Draw timeline of circuit with conventional gates
draw_timeline(
    ising_circuit_conventional,
    idle_wires=False,
    target=backend_conventional.target,
    time_range=(0, 500),
    style=IQXSimple(),
)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/0013f5fa-4072-4aa4-94fe-7e195435f828-0.svg" alt="Output of the previous code cell" />

In [4]:
# Draw timeline of circuit with fractional gates
draw_timeline(
    ising_circuit_fractional,
    idle_wires=False,
    target=backend_fractional.target,
    time_range=(0, 500),
    style=IQXSimple(),
)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/08dd1cdf-8b34-47c2-8324-f3538c9d1ab6-0.svg" alt="Output of the previous code cell" />

### Angle constraints

For the two-qubit $R_{ZZ}(\theta)$ gate, only angles between $0$ and $\pi/2$ can be executed on IBM Quantum hardware. If a circuit contains any $R_{ZZ}(\theta)$ gates with an angle outside of this range, then the standard transpilation pipeline will generally correct this with an appropriate circuit transformation (through the [`FoldRzzAngle`](../api/qiskit-ibm-runtime/transpiler-passes-fold-rzz-angle) pass).  However, for any $R_{ZZ}(\theta)$ gate containing one or more [`Parameter`](../api/qiskit/qiskit.circuit.Parameter)s, the transpiler will assume these parameters will be assigned angles within this range at runtime. The job will fail if any of the parameter values specified in the PUB submitted to Qiskit Runtime are outside of this range.

## Where to use fractional gates

Historically, the basis gates available on IBM Quantum QPUs have been **`CZ`**, **`X`**, **`RZ`**, **`SX`**, and **`ID`**, which can not efficiently represent circuits with single- and two-qubit rotations that are not multiples of $\pi / 2$. For example, an $R_X(\theta)$ gate, when transpiled, must decompose into a series of $RZ$ and $\sqrt{X}$ gates, which creates a circuit with two gates of finite duration instead of one.

Similarly, when two-qubit rotations such as an $R_{ZZ}(\theta)$ gate are transpiled, the decomposition requires two **`CZ`** gates and several single-qubit gates, which increases the circuit depth. These decompositions are shown in the following code.

In [5]:
qc = QuantumCircuit(1)
param = Parameter("θ")
qc.rx(param, 0)
qc.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/11b003ee-aa8e-4794-a84a-b6870d15fa11-0.svg" alt="Output of the previous code cell" />

In [6]:
# Decomposition of an RX(θ) gate using the IBM Quantum QPU basis
service = QiskitRuntimeService()
backend = service.backend("ibm_torino")
optimization_level = 3
pm = generate_preset_pass_manager(optimization_level, backend=backend)
transpiled_circuit = pm.run(qc)
transpiled_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/5da9bba4-9a3b-4569-9997-c5b9ccf87b6a-0.svg" alt="Output of the previous code cell" />

In [7]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService

qc = QuantumCircuit(2)
param = Parameter("θ")
qc.rzz(param, 0, 1)
qc.draw("mpl")

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/6b256201-0237-4f63-91ff-fde1bb884804-0.svg" alt="Output of the previous code cell" />

In [8]:
# Decomposition of an RZZ(θ) gate using the IBM Quantum QPU basis
service = QiskitRuntimeService()
backend = service.backend("ibm_torino")
optimization_level = 3
pm = generate_preset_pass_manager(optimization_level, backend=backend)
transpiled_circuit = pm.run(qc)
transpiled_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/fractional-gates/extracted-outputs/f07217b9-a6f0-4adf-b341-6da447535c33-0.svg" alt="Output of the previous code cell" />

For workflows that require many single-qubit $R_X(\theta)$ or two-qubit rotations (such as in a variational ansatz or when simulating the time evolution of quantum systems), this constraint causes the circuit depth to grow rapidly. However, fractional gates remove this requirement, because the single- and two-qubit rotations are executed directly, and create a more efficient (and thus error-suppressed) quantum circuit.

<span id="when-not-to-use"></span>
## When *not* to use fractional gates

It is important to note that fractional gates are an *experimental* feature and the behavior of the `use_fractional_gates` flag may change in the future. Look to the [release notes](/docs/api/qiskit-ibm-runtime/release-notes) for new versions of Qiskit Runtime for more information. See also the API reference documentation for [QiskitRuntimeService.backend](/docs/api/qiskit-ibm-runtime/qiskit-runtime-service#backend), which describes `use_fractional_gates`.

Additionally, the Qiskit transpiler has limited capability to use $R_{ZZ}(\theta)$ in its optimization passes. This requires you to take more care in crafting and optimizing circuits that contain these instructions.

Lastly, using fractional gates is not supported for:
- [Dynamic circuits](/docs/guides/classical-feedforward-and-control-flow)
- [Pauli twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) - however, [measurement twirling with TREX](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) *is* supported.
- [Probabilistic error cancellation](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)
- [Zero-noise extrapolation (using probabilistic error amplification)](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea)

Read the guide on [primitive options](/docs/guides/runtime-options-overview) to learn more about customizing the error mitigation and suppression techniques for a given quantum workload.

## Next steps

<Admonition type="tip" title="Recommendations">
  -  To learn more about transpilation, see the [introduction to transpilation](/docs/guides/transpile) page.
  -  Read about [writing a custom transpiler pass](/docs/guides/custom-transpiler-pass).
  -  Understand how to [configure error mitigation](/docs/guides/configure-error-mitigation) for Qiskit Runtime.
</Admonition>